In [1]:
from typing import List

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
from dotenv import load_dotenv
load_dotenv()

from speechfulagent.dataclasses import Experience

torch.cuda.is_available()

False

In [2]:
episode = [Experience(state=0, action=3, reward=0.0, next_state=0, done=False),
 Experience(state=0, action=3, reward=0.0, next_state=1, done=False),
 Experience(state=1, action=3, reward=0.0, next_state=0, done=False),
 Experience(state=0, action=3, reward=0.0, next_state=0, done=False),
 Experience(state=0, action=3, reward=0.0, next_state=0, done=False),
 Experience(state=0, action=3, reward=0.0, next_state=1, done=False),
 Experience(state=1, action=3, reward=0.0, next_state=2, done=False),
 Experience(state=2, action=0, reward=0.0, next_state=6, done=False),
 Experience(state=6, action=2, reward=0.0, next_state=10, done=False),
 Experience(state=10, action=2, reward=0.0, next_state=14, done=False),
 Experience(state=14, action=3, reward=0.0, next_state=10, done=False),
 Experience(state=10, action=2, reward=0.0, next_state=14, done=False),
 Experience(state=14, action=3, reward=1.0, next_state=15, done=True)]

In [3]:
class StateEncoder(nn.Module):
    def __init__(
        self,
        device: str="cuda"
    ):
        super().__init__()
        self.device = device

        self.state_mod = nn.Linear(16, 32)
        self.action_mod = nn.Linear(4, 32)
        self.reward_mod = nn.Linear(1, 32)

        self.lstm = nn.LSTM(input_size=96, hidden_size=512, batch_first=True)
        self.norm = nn.LayerNorm(512)
        self.project = nn.Linear(512, 2048)

    def forward(self, episode: List[Experience]):
        states = torch.as_tensor([exp.state for exp in episode], dtype=torch.long).to(self.device)
        states = self.state_mod(F.one_hot(states, 16).to(dtype=torch.float32))

        actions = torch.as_tensor([exp.action for exp in episode], dtype=torch.long).to(self.device)
        actions = self.action_mod(F.one_hot(actions, 4).to(dtype=torch.float32))

        rewards = torch.as_tensor([exp.reward for exp in episode], dtype=torch.float32).to(self.device)
        rewards = self.reward_mod(rewards.view((-1, 1)))

        exp_tensor = torch.concat([states, actions, rewards], dim=1)
        
        lstm_out, (h_n, c_n) = self.lstm(exp_tensor)
        normed = self.norm(lstm_out)
        projected = self.project(normed)
        return projected


In [11]:
# se = StateEncoder().to("cuda")
se = StateEncoder(device="cpu")
with open("weights.pth", "rb") as f:
    state_dict = torch.load(f, map_location=torch.device('cpu'))
se.load_state_dict(state_dict)

<All keys matched successfully>

In [12]:
sum([p.numel() for p in se.parameters()])

2301696

In [7]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B").to("cuda")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B")

# tokenizer = AutoTokenizer.from_pretrained("C:/Users/HONOR/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e")
# model = AutoModelForCausalLM.from_pretrained("C:/Users/HONOR/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [21]:
class Explainer():
    def __init__(self):
        self.transformer = None

    def _apply_top_k(self, logits: torch.Tensor, k: int=0) -> torch.Tensor:
        if k == 0:
            return logits
        top_logits, top_ids = torch.topk(logits, k, dim=-1)
        mask = torch.ones_like(logits, dtype=torch.bool)
        mask[top_ids] = False
        return torch.masked_fill(logits, mask=mask, value=float("-inf"))
    
    def _apply_temperature(self, logits: torch.Tensor, temperature: float) -> torch.Tensor:
        return logits / temperature
    
    def _greedy_sampling(self, logits: torch.Tensor) -> int:
        val, ind = torch.max(logits, dim=0)
        return int(ind.item())
    
    def _random_sampling(self, logits: torch.Tensor) -> int:
        probs = torch.softmax(logits, dim=-1)
        return int(torch.multinomial(probs, 1).item())
    
    def generate(
        self,
        input_embeds: torch.Tensor,
        startage,
        endage,
        startas,
        max_length: int=32,
        temperature: float=0.0,
        top_k: int=0
    ) -> str:
        if self.transformer is None:
            raise RuntimeError("Model not loaded!")
        
        tokens = []
        cache = None
        input_embeds = torch.cat([
            startage.squeeze(0),
            input_embeds.squeeze(0),
            endage.squeeze(0),
            startas.squeeze(0)
        ]).unsqueeze(0)
        for _ in range(max_length):
            with torch.no_grad():
                outputs = self.transformer.forward(inputs_embeds=input_embeds.unsqueeze(0).to(torch.bfloat16), cache=cache, use_cache=False)
                next_token_logits = outputs.logits[0, -1, -1, :]
                cache = outputs.past_key_values         

                if temperature == 0.0:
                    next_token = self._greedy_sampling(next_token_logits)
                else:
                    next_token_logits = self._apply_temperature(next_token_logits, temperature)
                    next_token_logits = self._apply_top_k(next_token_logits, top_k)
                    next_token = self._random_sampling(next_token_logits)
                tokens.append(next_token)
                input_embeds = torch.cat([input_embeds.squeeze(0), self.transformer.model.embed_tokens(torch.LongTensor([[next_token]]).to("cuda")).squeeze(0)]).unsqueeze(0)
        return input_embeds, tokens
    
    def generate_with_loss(
        self,
        input_embeds: torch.Tensor,
        startage,
        endage,
        startas,
        ground_truth,
        max_length: int=32,
        temperature: float=0.0,
        top_k: int=0,
    ):
        if self.transformer is None:
            raise RuntimeError("Model not loaded!")
        
        tokens = []
        cache = None
        loss = []
        
        prefix_len = startage.shape[1] + input_embeds.shape[1] + endage.shape[1] + startas.shape[1]

        ignore_index = -100
        full_labels = torch.full((prefix_len + len(ground_truth),), ignore_index, dtype=torch.long, device=ground_truth.device)
        full_labels[prefix_len:] = ground_truth
        
        current_input = torch.cat([
            startage.squeeze(0), 
            input_embeds.squeeze(0),
            endage.squeeze(0),
            startas.squeeze(0)
        ]).unsqueeze(0)
        
        for i in range(max_length):
            current_labels = full_labels[:current_input.shape[1]].unsqueeze(0)

            outputs = self.transformer.forward(
                inputs_embeds=current_input.to(torch.bfloat16), 
                labels=current_labels, 
                cache=cache, 
                use_cache=False
            )

            if outputs.loss is not None:
                loss.append(outputs.loss)

            next_token_logits = outputs.logits[0, -1, :]
            cache = outputs.past_key_values

            if temperature == 0.0:
                next_token = self._greedy_sampling(next_token_logits)
            else:
                next_token_logits = self._apply_temperature(next_token_logits, temperature)
                next_token_logits = self._apply_top_k(next_token_logits, top_k)
                next_token = self._random_sampling(next_token_logits)
            tokens.append(next_token)

            next_token_embed = self.transformer.model.embed_tokens(
                torch.LongTensor([[next_token]])
            ).squeeze(0)

            current_input = torch.cat([current_input.squeeze(0), next_token_embed]).unsqueeze(0)
        return current_input, tokens, loss

In [22]:
explainer = Explainer()
explainer.transformer = model
explainer.transformer
se.train()

se_optim = optim.Adam(se.parameters(), lr=0.001)

In [17]:
explanation = "Агент использовал среду для того, чтобы проскользить по верхней части поля. Затем он успешно добрался до подарка."

In [18]:
tok_exp = tokenizer.encode(explanation)
print(len(tok_exp))

35


In [ ]:
with torch.no_grad():
    startage = explainer.transformer.model.embed_tokens(torch.LongTensor([[151644, 872, 198]]).to("cuda"))
    endage = explainer.transformer.model.embed_tokens(torch.LongTensor([[151645, 198]]).to("cuda"))
    startas = explainer.transformer.model.embed_tokens(torch.LongTensor([[151644, 77091, 198]]).to("cuda"))

In [23]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(se_optim, 100)

In [13]:
def hook(module, input, output):
    print(f"Hook called! Module: {module.__class__.__name__}")
    print(f"Grad norm: {module.all_weights[0][0].grad.norm()}")

handler = se.lstm.register_full_backward_hook(hook)
se.train()

StateEncoder(
  (state_mod): Linear(in_features=16, out_features=32, bias=True)
  (action_mod): Linear(in_features=4, out_features=32, bias=True)
  (reward_mod): Linear(in_features=1, out_features=32, bias=True)
  (lstm): LSTM(96, 512, batch_first=True)
  (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (project): Linear(in_features=512, out_features=2048, bias=True)
)

In [24]:
for it in range(100):
    se_optim.zero_grad()

    outputs = se.forward(episode)

    _, tokens, loss = explainer.generate_with_loss(
        outputs.unsqueeze(0).to(dtype=torch.bfloat16), 
        startage, endage, startas,
        torch.LongTensor(tok_exp).to("cuda"),
        max_length=35, 
        temperature=0.6)
    
    if not loss:
        continue
    sum_loss = sum(loss[1:])

    sum_loss.backward()
    print(f"{it} iter, Loss: {sum_loss.item()}")
    print(tokenizer.decode(tokens))
    se_optim.step()
    scheduler.step()

Hook called! Module: LSTM
Grad norm: 0.05403069034218788
0 iter, Loss: 2.0287671089172363
Агент использовал среду для того, чтобы проскользить по верхней части поля. Затем он успешно добрался до до успешнообр
Hook called! Module: LSTM
Grad norm: 0.046975910663604736
1 iter, Loss: 1.785407304763794
Агент использовал среду для того, чтобы проскользить по верхней части поля. Затем он успешно добрался до ону успешно
Hook called! Module: LSTM
Grad norm: 0.043172720819711685
2 iter, Loss: 1.5159671306610107
Агент использовал среду для того, чтобы проскользить по верхней части поля. Затем он успешно добрался до части дообр
Hook called! Module: LSTM
Grad norm: 0.03821516036987305
3 iter, Loss: 1.337218165397644
Агент использовал среду для того, чтобы проскользить по верхней части поля. Затем он успешно добрался до добрить
Hook called! Module: LSTM
Grad norm: 0.03746698796749115
4 iter, Loss: 1.1906574964523315
Агент использовал среду для того, чтобы проскользить по верхней части поля. Затем он

KeyboardInterrupt: 

In [26]:
with open("weights.pth", "wb") as f:
    torch.save(se.state_dict(), f)

In [24]:
from speechfulagent.agent import A2CAgent
import gymnasium as gym

env = gym.wrappers.RecordVideo(gym.make("FrozenLake-v1", render_mode="rgb_array"), "videos")
agent = A2CAgent(env)
agent.load_model("agent_models")
agent.eval()

d:\Reality\Учёба\Диплом\SpeechfulAgent\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:293: UserWarning: WARN: Overwriting existing videos at d:\Reality\Учёба\Диплом\SpeechfulAgent\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [58]:
episode = []
agent.reset()
while True:
    exp = agent.step()
    episode.append(exp)
    if exp.done:
        break

In [60]:
flag = False
while not flag:
    episode = []
    agent.reset()
    while True:
        exp = agent.step()
        episode.append(exp)
        if exp.done:
            if exp.reward == 0.0:
                flag = True
            break

In [61]:
episode

[Experience(state=0, action=3, reward=0.0, next_state=0, done=False),
 Experience(state=0, action=3, reward=0.0, next_state=1, done=False),
 Experience(state=1, action=3, reward=0.0, next_state=0, done=False),
 Experience(state=0, action=3, reward=0.0, next_state=0, done=False),
 Experience(state=0, action=3, reward=0.0, next_state=0, done=False),
 Experience(state=0, action=2, reward=0.0, next_state=4, done=False),
 Experience(state=4, action=3, reward=0.0, next_state=5, done=True)]

In [62]:
with torch.no_grad():
    outputs = se.forward(episode)

    _, tokens, loss = explainer.generate_with_loss(
        outputs.unsqueeze(0).to(dtype=torch.bfloat16), 
        startage, endage, startas,
        torch.LongTensor(tok_exp),
        max_length=35, 
        temperature=0.6)
tokenizer.decode(tokens)

'Агент использовал среду для того, чтобы ответить на вопрос. Он успешно далал далкобрем Золкемену. и.'

In [15]:
outputs, (h_n, c_n) = se.forward(episode)
_, tokens= explainer.generate(
    outputs.unsqueeze(0).to(dtype=torch.bfloat16), 
    startage,   
    endage,
    startas,
    max_length=64, 
    temperature=0.0
)
tokenizer.batch_decode(tokens)[0]

ValueError: too many values to unpack (expected 2)